# 数字水印综述 — 自学课程

**分类**：数字水印

**内容简介**：涵盖扩频水印嵌入提取、强度优化、擦除还原及实用工具与实验



## 学习目标

- 理解水印系统的目标与指标
- 能实现最小可用的嵌入/提取
- 能模拟攻击通道并评估鲁棒性
- 能解释常见设计取舍



## 符号与抽象

$$x' = \mathrm{Embed}(x,w,k)$$
$$\hat{w} = \mathrm{Extract}(x',k)$$

其中：
- $x$：载体（carrier）
- $w$：水印信息
- $k$：密钥/种子/参数



## 自检小测

1) 不可感知性与鲁棒性之间通常是什么关系？
2) 为什么很多水印方案需要密钥 k？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



In [ ]:
# Step 1：基础指标 MSE/PSNR（纯 Python）
import math, random, secrets

def mse(x, y):
    if len(x)!=len(y):
        raise ValueError("长度不一致")
    return sum((a-b)**2 for a,b in zip(x,y))/len(x)

def psnr(x, y, max_val=255.0):
    m=mse(x,y)
    if m==0:
        return float('inf')
    return 10*math.log10((max_val**2)/m)

x=[10,20,30,40,50]
y=[10,21,29,41,49]
print("MSE=", mse(x,y))
print("PSNR=", round(psnr(x,y),2))
# 预期输出：MSE 为小数，PSNR 为正数



# Step 2：最小可用示例——LSB 水印

把比特写入最低有效位：

$$x'_i=(x_i\&\sim 1)\;|\;w_i$$

提取：

$$\hat{w}_i=x'_i\&1$$



In [ ]:
# Step 2：比特/文本互转
def bits_from_text(s: str):
    b=s.encode("utf-8")
    out=[]
    for byte in b:
        for i in range(7,-1,-1):
            out.append((byte>>i)&1)
    return out

def text_from_bits(bits):
    if len(bits)%8!=0:
        raise ValueError("bits 长度需为 8 的倍数")
    out=bytearray()
    for i in range(0,len(bits),8):
        byte=0
        for j in range(8):
            byte=(byte<<1)|bits[i+j]
        out.append(byte)
    return out.decode("utf-8", errors="replace")

print(bits_from_text("A")[:8], text_from_bits(bits_from_text("A")))
# 预期输出：([0,1,0,0,0,0,0,1], 'A')



In [ ]:
# Step 3：LSB 嵌入与提取
def lsb_embed(carrier, bits):
    if len(bits)>len(carrier):
        raise ValueError("载体不足")
    out=carrier[:]
    for i,b in enumerate(bits):
        out[i]=(out[i]&~1)|int(b)
    return out

def lsb_extract(stego, n_bits):
    return [(stego[i]&1) for i in range(n_bits)]

carrier=[secrets.randbelow(256) for _ in range(300)]
wm=bits_from_text("WM")
stego=lsb_embed(carrier, wm)
wm_hat=lsb_extract(stego, len(wm))
print(text_from_bits(wm_hat))
print("PSNR=", round(psnr(carrier, stego),2))
# 预期输出：WM 与一个较大的 PSNR



# Step 4：攻击通道与 BER

我们用 BER 衡量比特错误率：

$$\mathrm{BER}=\frac{\#\{i:w_i\neq\hat{w}_i\}}{n}$$



In [ ]:
# Step 4：BER 与噪声攻击
def ber(bits, bits_hat):
    wrong=sum(1 for a,b in zip(bits,bits_hat) if a!=b)
    return wrong/len(bits)

def add_noise(arr, p=0.1):
    out=arr[:]
    for i in range(len(out)):
        if random.random()<p:
            out[i]=max(0, min(255, out[i]+(1 if random.random()<0.5 else -1)))
    return out

attacked=add_noise(stego, p=0.2)
wm_hat2=lsb_extract(attacked, len(wm))
print("BER=", ber(wm, wm_hat2))
# 预期输出：BER 往往 > 0



# Step 5：增强：重复编码（三次投票）

一个最简单的纠错：每个比特重复 3 次。提取时对 3 个结果做多数投票。



In [ ]:
# Step 5：三次重复编码
def repeat3(bits):
    out=[]
    for b in bits:
        out.extend([b,b,b])
    return out

def vote3(bits):
    if len(bits)%3!=0:
        raise ValueError("长度需为 3 的倍数")
    out=[]
    for i in range(0,len(bits),3):
        out.append(1 if sum(bits[i:i+3])>=2 else 0)
    return out

wm3=repeat3(wm)
stego3=lsb_embed(carrier, wm3)
attacked3=add_noise(stego3, p=0.2)
wm_hat3=vote3(lsb_extract(attacked3, len(wm3)))
print("BER_no_ec=", ber(wm, lsb_extract(attacked, len(wm))))
print("BER_ec=", ber(wm, wm_hat3))
# 预期输出：BER_ec 往往小于 BER_no_ec（但不保证每次都更小）



## 练习

1) 扫描不同噪声概率 p，画出（用打印表格即可）BER 的变化。
2) 把载体换成更小范围（0..31），观察 PSNR 变化。
3) 思考：为什么频域水印往往更鲁棒？尝试用“压缩会丢掉哪些信息”解释。


## 总结与延伸

你已经完成一个最小可用水印系统：嵌入→攻击→提取→评估。

> 下一步建议：学习频域嵌入、同步码、纠错码，以及对抗几何攻击（裁剪/旋转）。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。

